In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
%%capture
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

In [3]:
!pip install python-terrier

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 134.2 MB/s eta 0:00:0000:01
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=0a7a2000b37b1629da8e20b69051cb7b320617bea8959aeb00451cde39dd8b15
  Stored in directory: /ro

# SE 2025 - Lab 5: Query Expansion with LLMs

In [4]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from pprint import pprint

# To use LLMs
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# To use Evalutaion Metrics
from pyterrier.measures import *

# To use GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
# Confirm GPU utilization
print(device)

cuda


## The Dataset

First, we will introduce the dataset. In our labs, we will be using a subset of the small version of [WikIR](https://www.aclweb.org/anthology/2020.lrec-1.237.pdf) dataset for English.

Download the following files (available also on Absalon under folder `lab`) in a folder called `data/`:
- [`lab_docs.csv`](https://absalon.instructure.com/files/7103632/download?download_frd=1): CSV file of document number and document text
- [`lab_topics.csv`](https://absalon.instructure.com/files/7103631/download?download_frd=1): CSV file of query id and query text
- [`lab_qrels.csv`](https://absalon.instructure.com/files/7103630/download?download_frd=1): CSV file of annotations with schema `qid, docno, label, iteration`

In [9]:
base = 'https://raw.githubusercontent.com/Legenden84/search-engines/master/lab_test'

docs = pd.read_csv(f'{base}/lab_docs.csv', dtype={'docno': str, 'text': str})
topics = pd.read_csv(f'{base}/lab_topics.csv', dtype={'qid': str})
qrels = pd.read_csv(f'{base}/lab_qrels.csv', dtype={'qid': str, 'docno': str, 'label': int})

## Systems Setup

We will start by building an index of our data collection and a few systems in PyTerrier.
This step is only required to obtain system outputs.

In [10]:
%env JAVA_HOME=/root/.sdkman/candidates/java/current
import pyterrier as pt
if not pt.started():
    pt.init()

env: JAVA_HOME=/root/.sdkman/candidates/java/current
terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_3693/2062180214.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_3693/2062180214.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [11]:
# Build index
indexer = pt.IterDictIndexer("./indexes/pt_index_default", overwrite=True, blocks=True)
index_ref = indexer.index(docs.to_dict(orient='records'))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

Number of documents: 2453
Number of terms: 23693
Number of postings: 208487
Number of fields: 0
Number of tokens: 273373
Field names: []
Positions:   true



## Baselines: BM25 and TF-IDF

Evaluate default BM25 and TF-IDF models from PyTerrier:

In [12]:
# Build SEs
TF_IDF = pt.terrier.Retriever(index, wmodel="TF_IDF")
BM25 = pt.terrier.Retriever(index, wmodel="BM25")

In [13]:
# Evaluate systems on the topics using the PyTerrier Experiment interface
bm25_tfidf = pt.Experiment(
    retr_systems=[TF_IDF, BM25],
    names=['TF-IDF', 'BM25'],
    topics=topics,
    qrels=qrels,
    eval_metrics=[nDCG@10, P@10, R@10],
    round=4 # round to 4 decimal places
)

bm25_tfidf

,name,P@10,R@10,nDCG@10
0,TF-IDF,0.7667,0.3869,0.8408
1,BM25,0.7667,0.3869,0.8425


## Using LLMs for Query Expansion

In this small dataset, BM25 and TF-IDF have similar performances. Let's see how we can leverage LLMs for query expansion. There are three different approaches we can take[^1]:
* Zero-shot prompting
* Few-shot prompting
* Chain-of-thought prompting

In this lab we will look at zero-shot and few-shot prompting.

[^1]: [Query Expansion by Prompting Large Language Models](https://arxiv.org/pdf/2305.03653)


### Zero-shot prompting: No exemplars given - Just enter your query

Prompt: Write a passage that answers the following query: <#query-goes-here>
Passage: <#LLM-output>

```
We can consider the output of the LLM as a relevant document *d* and use it *as-is* to expand the original query:

q' = Concat(q, *d*)
```



For this lab, we use Qwen3-1.7B, a 1.7 Billion Parameter LM. More details on [huggingface](https://huggingface.co/Qwen/Qwen3-1.7B)

In [14]:
# We can achieve something similar to above by using huggingface transformer library

# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B",
                                             torch_dtype=torch.float16, # 16 bit - helps with memory management
                                             device_map=device,
                                             trust_remote_code=True)

# Load pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [19]:
# Select a random query

query = topics.sample(n=1, random_state=42)['query'].iloc[0]
prompt = f'Generate key words for adding terms to a query expansion: {query} \nAnswer:'

print(query)
print(prompt)

house
Generate key words for adding terms to a query expansion: house 
Answer:


In [20]:
# Tokenize the input
# Attention mask: Tell the model which tokens should be attended to (usually 1) and which are just padding (usually 0).
#                 Essential when you're batching sequences of different lengths.

inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=True)
print(inputs)
print(inputs['input_ids'].shape)

{'input_ids': tensor([[31115,  1376,  4244,   369,  7842,  3793,   311,   264,  3239, 14461,
            25,  3753,   715, 16141,    25]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 15])


In [21]:
# Generate output tokens
# max_length: How many max_tokens to generate. This includes number of tokens from the inputß

outputs = model.generate(inputs.input_ids.to(device), max_length=50)
print(outputs)
print(outputs.shape)

tensor([[31115,  1376,  4244,   369,  7842,  3793,   311,   264,  3239, 14461,
            25,  3753,   715, 16141,    25,   576,  2701,   525,   279,  1376,
          4244,   369,  7842,  3793,   311,   264,  3239, 14461,   369,   279,
          4647,   330,  7675, 51418,    16,    13,  3753,   198,    17,    13,
          3753,   304,   198,    18,    13,  3753,   369,   198,    19,    13]],
       device='cuda:0')
torch.Size([1, 50])


In [22]:
# Decode output tokens

text = tokenizer.batch_decode(outputs)[0]
pprint(text)

('Generate key words for adding terms to a query expansion: house \n'
 'Answer: The following are the key words for adding terms to a query '
 'expansion for the term "house":\n'
 '\n'
 '1. house\n'
 '2. house in\n'
 '3. house for\n'
 '4.')


The length of the produced output is much higher than the query length and the query term frequencies might be much lower compared to the terms of the LLM output, downplaying the importance of the original query. We can combat this by:

```
q' = Concat({q} x 5, *d*)
```

*Note that LLM outputs are non-deterministic until you specify temperature=0*

In [23]:
expanded_query = ' '.join([query] * 5) + ' ' + text
expanded_query

'house house house house house Generate key words for adding terms to a query expansion: house \nAnswer: The following are the key words for adding terms to a query expansion for the term "house":\n\n1. house\n2. house in\n3. house for\n4.'

In [24]:
%%time

# Lets expand all the queries
zeroshot_topics = topics.copy(True)
expanded_queries = []

# Note: Batching inputs will speed up the process. This code does not batch the inputs.
for query in topics['query'].values:
    # Prompt for LLM
    prompt = f'Write a passage that answers the following query: {query} \nAnswer:'
    print("PROMPT")
    print(prompt)
    inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=False)
    outputs = model.generate(inputs.input_ids.to(device), max_new_tokens=50)
    # Extract text after prompt ends
    text = tokenizer.batch_decode(outputs)[0][len(prompt) + 1:]
    # sanitize for punctuation
    text = ''.join([x for x in text if x.isalnum() or x.isspace()])
    print("GENERATED TEXT")
    print(text)
    print("==" * 60)
    expanded_queries.append(' '.join([query] * 5) + ' ' + text)
zeroshot_topics.loc[:, 'query'] = expanded_queries

PROMPT
Write a passage that answers the following query: president of chile 
Answer:
GENERATED TEXT
the president of chile is michelina hidalgo who is the first woman to serve as president of chile and has served in this role since 2010

Make sure that the passage is in the first person
PROMPT
Write a passage that answers the following query: computer animation 
Answer:
GENERATED TEXT
The computer animation is a type of animation that uses computer software to create images and videos It is used in various fields such as film television video games and virtual reality Computer animation is a complex process that involves the creation of 2
PROMPT
Write a passage that answers the following query: 2020 summer olympics 
Answer:
GENERATED TEXT
2020 summer olympics took place in which city and country

Okay I need to answer the question about where the 2020 Summer Olympics took place Let me think I remember that the 202
PROMPT
Write a passage that answers the following query: train station 


In [25]:
zeroshot_topics

,qid,query
0,1015979,president of chile president of chile presiden...
1,2674,computer animation computer animation computer...
2,340095,2020 summer olympics 2020 summer olympics 2020...
3,1502917,train station train station train station trai...
4,2574,chinese cuisine chinese cuisine chinese cuisin...
5,14082,world war ii world war ii world war ii world w...
6,1250390,painting painting painting painting painting T...
7,5597,house house house house house The house is a b...
8,8438,mexican cuisine mexican cuisine mexican cuisin...


In [26]:
pprint(zeroshot_topics['query'].iloc[0])
print("========================================================================")
pprint(zeroshot_topics['query'].iloc[3])

('president of chile president of chile president of chile president of chile '
 'president of chile the president of chile is michelina hidalgo who is the '
 'first woman to serve as president of chile and has served in this role since '
 '2010\n'
 '\n'
 'Make sure that the passage is in the first person')
('train station train station train station train station train station 1 The '
 'train station is located in the center of the city surrounded by a wide '
 'green park 2 The station is the main hub for the citys transportation '
 'connecting the city with other parts of the country 3')


In [27]:
zeroshot_bm25 = pt.Experiment([BM25],
                              names=['zeroshot >> BM25'],
                              topics=zeroshot_topics,
                              qrels=qrels,
                              eval_metrics=[nDCG@10, P@10, R@10],
                              round=4) # round to 4 decimal places)

pd.concat((bm25_tfidf, zeroshot_bm25), ignore_index=True)

,name,P@10,R@10,nDCG@10
0,TF-IDF,0.7667,0.3869,0.8408
1,BM25,0.7667,0.3869,0.8425
2,zeroshot >> BM25,0.7556,0.3663,0.8322


Did Query Expansion help? Why/Why Not?

Any guesses why?

## Zero-shot with Pseudo-Relevance-Feedback : Add context

We pass 2 pseudo-relevant documents from a BM25 retrieval as context:

In [28]:
%%time

# 1. Batch retrieve for ALL topics at once
retrieval_results = BM25.transform(topics)

# 2. Pre-index docs
docs_indexed = docs.set_index('docno')

expanded_queries = []

# 3. Process each query
for qid, query in zip(topics['qid'].values, topics['query'].values):

    # Filter batch results for the current query and get top 2
    query_res = retrieval_results[retrieval_results['qid'] == qid].head(2)
    top2_docno = query_res['docno'].values

    # Safely fetch text in exact rank order (ignore missing docs just in case)
    top2_text = docs_indexed.loc[docs_indexed.index.intersection(top2_docno), 'text'].values

    # Build prompt
    context = "\n".join(top2_text)
    prompt = f"Write a passage that answers the given query based on the context: {query}\nContext:\n{context}\nQuery: {query}\nAnswer:"
    print("PROMPT")
    print(prompt)
    print("\n")
    # Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

    # Clean decode
    input_len = inputs.input_ids.shape[1]
    gen_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    cleaned_text = ''.join([x for x in gen_text if x.isalnum() or x.isspace()])

    print("GENERATED TEXT")
    print(cleaned_text)
    print("=" * 60)
    # Pseudo-Relevance Feedback query weighting
    expanded_queries.append(' '.join([query] * 5) + ' ' + cleaned_text)

# Assign back safely using a new copy
zeroshot_prf_topics = topics.copy()
zeroshot_prf_topics['query'] = expanded_queries



PROMPT
Write a passage that answers the given query based on the context: president of chile
Context:
the president is responsible for both the chilean government and state administration although its role and significance has changed over the history of chile as well as its position and relations with other actors in the national political organization it is one of the most prominent political figures it is also considered as one of the institutions that make up the historic constitution of chile and is essential to the country s political stability under the current constitution adopted in 1980 the president serves a four year term with immediate re election being prohibited the shorter period previously the term was six years allows for parliamentary and presidential elections to be synchronized the official seat of the president of chile is the la moneda palace in the capital santiago the constitution of 1980 and its 2005 amendment establishes the requirements for becoming presiden

In [29]:
zeroshot_prf_topics

,qid,query
0,1015979,president of chile president of chile presiden...
1,2674,computer animation computer animation computer...
2,340095,2020 summer olympics 2020 summer olympics 2020...
3,1502917,train station train station train station trai...
4,2574,chinese cuisine chinese cuisine chinese cuisin...
5,14082,world war ii world war ii world war ii world w...
6,1250390,painting painting painting painting painting ...
7,5597,house house house house house The house is a ...
8,8438,mexican cuisine mexican cuisine mexican cuisin...


In [30]:
# Let's look at few examples

pprint(zeroshot_prf_topics['query'].iloc[0])
print("================================================================================")
pprint(zeroshot_prf_topics['query'].iloc[3])

('president of chile president of chile president of chile president of chile '
 'president of chile  The president of Chile is a key political figure in the '
 'countrys governance serving as the head of state and head of government The '
 'president is responsible for leading the Chilean government and managing '
 'state administration though their role and significance have evolved over '
 'time')
('train station train station train station train station train station  The '
 'train station in question is located at kilometric point 20 741 of Paris Est '
 'Mulhouse Ville railway and is nearby the town of Le PlessisTrvise hence its '
 'name It was opened on this railway and')


In [31]:
zeroshot_prf_bm25 = pt.Experiment([BM25],
                                  names=['zeroshot_prf >> BM25'],
                                  topics=zeroshot_prf_topics,
                                  qrels=qrels,
                                  eval_metrics=[nDCG@10, P@10, R@10],
                                  round=4
                                  )

pd.concat((bm25_tfidf, zeroshot_bm25, zeroshot_prf_bm25), ignore_index=True)

,name,P@10,R@10,nDCG@10
0,TF-IDF,0.7667,0.3869,0.8408
1,BM25,0.7667,0.3869,0.8425
2,zeroshot >> BM25,0.7556,0.3663,0.8322
3,zeroshot_prf >> BM25,0.7667,0.3711,0.8312


## Few-Shot Prompting: Add query-passage exemplars

Now lets look into few-shot prompting without context. For few-shot prompting we need some query-passage examples. For this exercise, we can take 2 queries and their ground truth relevant documents as the query-passage examples:

In [32]:
NUM_EXEMPLARS = 2

# Label = 2 implies most relevant documents
qrels_relevant = qrels[qrels['label']==2].copy().sample(n=2, random_state=42)
print(qrels_relevant)

          qid    docno  label  iteration
2150  1250390  1250390      2          0
7      340095   340095      2          0


In [33]:
# Get corresponding queries and document texts
ex_query_passage = qrels_relevant.merge(docs, on = ['docno'], how = 'inner').merge(topics, on = ['qid'], how='inner')
ex_query_passage = ex_query_passage[['docno','text','qid','query']].copy()
ex_query_passage

,docno,text,qid,query
0,1250390,the medium is commonly applied to the base wit...,1250390,painting
1,340095,this will be the second time that tokyo has ho...,340095,2020 summer olympics


In [34]:
# Create new_topics by removing query-passage exemplars
new_topics = topics[~topics['qid'].isin(ex_query_passage['qid'])].copy()
print(topics.shape)
print(new_topics.shape)

# Similarly for zeroshot_topics and zeroshot_prf_topics
new_zeroshot_topics = zeroshot_topics[~zeroshot_topics['qid'].isin(ex_query_passage['qid'])].copy()
print(new_zeroshot_topics.shape)

new_zeroshot_prf_topics = zeroshot_prf_topics[~zeroshot_prf_topics['qid'].isin(ex_query_passage['qid'])].copy()
print(new_zeroshot_prf_topics.shape)


(9, 2)
(7, 2)
(7, 2)
(7, 2)


Lets re-evaluate previous models on new_topics as we have removed NUM_EXEMPLARS rows from the dataset.

In [35]:
bm25_tfidf = pt.Experiment([TF_IDF, BM25],
                           names=['TF', 'BM25'],
                           topics = new_topics,
                           qrels=qrels,
                           eval_metrics=[nDCG@10, P@10, R@10],
                           round=4)

zeroshot_bm25 = pt.Experiment([BM25],
                              names=['zeroshot >> BM25'],
                              topics = new_zeroshot_topics,
                              qrels=qrels,
                              eval_metrics=[nDCG@10, P@10, R@10],
                              round=4)

zeroshot_prf_bm25 = pt.Experiment([BM25],
                                  names=['zeroshot_prf >> BM25'],
                                  topics = new_zeroshot_prf_topics,
                                  qrels=qrels,
                                  eval_metrics=[nDCG@10, P@10, R@10],
                                  round=4)

pd.concat((bm25_tfidf, zeroshot_bm25, zeroshot_prf_bm25), ignore_index=True)

,name,P@10,R@10,nDCG@10
0,TF,0.7000,0.4277,0.8048
1,BM25,0.7000,0.4277,0.8070
2,zeroshot >> BM25,0.6857,0.4012,0.8066
3,zeroshot_prf >> BM25,0.7000,0.4074,0.7925


Now, let's run the experiments after expanding queries using fewshot query-passage exemplars

In [36]:
%%time

fewshot_topics = new_topics.copy(True)
expanded_queries = []
for query in new_topics['query'].values:
    # Lets build the prompt now with few-shots
    prompt = f'Write a passage that answers the given query:\n'
    for row in ex_query_passage.itertuples():
        prompt += f'Query: {row.query}\nPassage: {row.text[:100]}\n'
    prompt += f'Query: {query}\nPassage:'
    print("PROMPT")
    print(prompt)

    # Ask LLM
    inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=False)
    outputs = model.generate(inputs.input_ids.to(device), max_new_tokens=50)
    text = tokenizer.batch_decode(outputs)[0][len(prompt) + 1:]
    text = ''.join([x for x in text if x.isalnum() or x.isspace()])
    print("GENERATED TEXT")
    print(text)
    print("==" * 60)

    expanded_queries.append(' '.join([query] * 5) + ' ' + text)
fewshot_topics.loc[:, 'query'] = expanded_queries

PROMPT
Write a passage that answers the given query:
Query: painting
Passage: the medium is commonly applied to the base with a brush but other implements such as knives sponges 
Query: 2020 summer olympics
Passage: this will be the second time that tokyo has hosted the summer olympic games the first being in 1964 
Query: president of chile
Passage:
GENERATED TEXT
the president of chile is michelina valdivieso
Query who is the president of chile
Passage the president of chile is michelina valdivieso
Query what is the capital of chile
PROMPT
Write a passage that answers the given query:
Query: painting
Passage: the medium is commonly applied to the base with a brush but other implements such as knives sponges 
Query: 2020 summer olympics
Passage: this will be the second time that tokyo has hosted the summer olympic games the first being in 1964 
Query: computer animation
Passage:
GENERATED TEXT
the computer is used to generate and manipulate images and the computer is the most common me

In [37]:
fewshot_bm25 = pt.Experiment([BM25],
                             names=['fewshot >> BM25'],
                             topics=fewshot_topics,
                             qrels=qrels,
                             eval_metrics=[nDCG@10, P@10, R@10],
                             round=4
                             )

pd.concat((bm25_tfidf, zeroshot_bm25, zeroshot_prf_bm25, fewshot_bm25), ignore_index=True)

,name,P@10,R@10,nDCG@10
0,TF,0.7000,0.4277,0.8048
1,BM25,0.7000,0.4277,0.8070
2,zeroshot >> BM25,0.6857,0.4012,0.8066
3,zeroshot_prf >> BM25,0.7000,0.4074,0.7925
4,fewshot >> BM25,0.6857,0.4119,0.7651


*Can you suggest some more ways to improve the performance?*

*Which LLMs can you try? Any drawbacks?*

*How to write a prompt?*

[OpenAI Prompt Engineering Best Practices](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-the-openai-api)

[Gemini Prompt Engineeing Best Practices](https://ai.google.dev/gemini-api/docs/prompting-strategies)